In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!cp -r drive/MyDrive/PlantVillage /content/PlantVillage
path="/content/PlantVillage"

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, transforms
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import random

device = "cuda" if torch.cuda.is_available() else "cpu"


In [4]:
full_dataset = datasets.ImageFolder(root=path, transform=None)
targets = [sample[1] for sample in full_dataset.samples]
classes = list(full_dataset.class_to_idx.keys())


train_idx, test_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.2,
    random_state=42,
    stratify=targets
)

print(f"Dataset has {len(classes)} classes.")
print(f"Number of Train images: {len(train_idx)} | Number of Test images: {len(test_idx)}")


Dataset has 15 classes.
Number of Train images: 16567 | Number of Test images: 4142


In [5]:
temp_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

temp_full = datasets.ImageFolder(root=path, transform=temp_transform)
temp_train = Subset(temp_full, train_idx)

def get_mean_std_rgb(dataset, batch_size=32, num_workers=2, sample_size=2000):
    if sample_size and len(dataset) > sample_size:
        indices = random.sample(range(len(dataset)), sample_size)
        dataset = Subset(dataset, indices)

    loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)

    rgb_sum = torch.zeros(3)
    rgb_sumsq = torch.zeros(3)
    count_pixels = 0

    for x, _ in tqdm(loader, desc="Calculating RGB mean/std"):
        b, c, h, w = x.shape
        count_pixels += b * h * w
        rgb_sum += x.sum(dim=[0, 2, 3])
        rgb_sumsq += (x**2).sum(dim=[0, 2, 3])

    mean = rgb_sum / count_pixels
    std = torch.sqrt(rgb_sumsq / count_pixels - mean**2)
    return mean, std

mean_tensor, std_tensor = get_mean_std_rgb(temp_train, sample_size=3000)
mean, std = mean_tensor.tolist(), std_tensor.tolist()

print(f"Calculated Mean (RGB): {mean}")
print(f"Calculated Std (RGB): {std}")


Calculating RGB mean/std:   0%|          | 0/94 [00:00<?, ?it/s]

Calculated Mean (RGB): [0.4593556523323059, 0.4749314486980438, 0.41079649329185486]
Calculated Std (RGB): [0.1855364739894867, 0.1620500236749649, 0.19974268972873688]


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])


In [7]:
train_data_all = datasets.ImageFolder(root=path, transform=train_transform)
test_data_all = datasets.ImageFolder(root=path, transform=test_transform)

train_set = Subset(train_data_all, train_idx)
test_set = Subset(test_data_all, test_idx)

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of testing batches: {len(test_loader)}")


Number of training batches: 518
Number of testing batches: 130


In [8]:
class PlantVillageCNN_RGB(nn.Module):
    def __init__(self, output_shape):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block_3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=64 * 28 * 28, out_features=512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(in_features=512, out_features=output_shape)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.conv_block_3(x)
        x = self.classifier(x)
        return x

model = PlantVillageCNN_RGB(output_shape=len(classes)).to(device)


In [9]:
def train_step(model, dataloader, loss_fn, optimizer):
    model.train()
    train_loss, train_acc = 0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)
    return train_loss / len(dataloader), train_acc / len(dataloader)

def test_step(model, dataloader, loss_fn):
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            test_loss += loss_fn(y_pred, y).item()
            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
            test_acc += (y_pred_class == y).sum().item()/len(y_pred)
    return test_loss / len(dataloader), test_acc / len(dataloader)


In [10]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20

for epoch in tqdm(range(EPOCHS)):
    train_loss, train_acc = train_step(model, train_loader, loss_fn, optimizer)
    test_loss, test_acc = test_step(model, test_loader, loss_fn)
    print(
        f"Epoch: {epoch+1} | "
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_acc*100:.2f}% | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {test_acc*100:.2f}%"
    )


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch: 1 | Train loss: 1.1714 | Train acc: 61.73% | Test loss: 0.5956 | Test acc: 80.68%
Epoch: 2 | Train loss: 0.5883 | Train acc: 80.73% | Test loss: 0.3489 | Test acc: 89.16%
Epoch: 3 | Train loss: 0.4148 | Train acc: 86.25% | Test loss: 0.2851 | Test acc: 91.32%
Epoch: 4 | Train loss: 0.3386 | Train acc: 88.98% | Test loss: 0.2791 | Test acc: 91.18%
Epoch: 5 | Train loss: 0.2882 | Train acc: 90.35% | Test loss: 0.1998 | Test acc: 94.38%
Epoch: 6 | Train loss: 0.2693 | Train acc: 91.11% | Test loss: 0.1975 | Test acc: 94.38%
Epoch: 7 | Train loss: 0.2238 | Train acc: 92.66% | Test loss: 0.2085 | Test acc: 94.09%
Epoch: 8 | Train loss: 0.2004 | Train acc: 93.27% | Test loss: 0.3308 | Test acc: 89.82%
Epoch: 9 | Train loss: 0.2031 | Train acc: 93.34% | Test loss: 0.1807 | Test acc: 94.93%
Epoch: 10 | Train loss: 0.1762 | Train acc: 94.05% | Test loss: 0.1740 | Test acc: 95.89%
Epoch: 11 | Train loss: 0.1684 | Train acc: 94.34% | Test loss: 0.1896 | Test acc: 95.38%
Epoch: 12 | Train l

In [11]:
torch.save(model.state_dict(), "plantvillage_rgb_weights.pth")


In [12]:
model = PlantVillageCNN_RGB(output_shape=len(classes))
model.load_state_dict(torch.load("plantvillage_rgb_weights.pth", map_location=device))
model.to(device)
model.eval()


PlantVillageCNN_RGB(
  (conv_block_1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_2): Sequential(
    (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_block_3): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=50176, out_features=512, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=512, out_features=15, bias=True)
  )
)

In [13]:
from google.colab import files
files.download("plantvillage_rgb_weights.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
from PIL import Image
from torchvision import transforms
import torch


In [18]:
image_path = "/content/PlantVillage/Potato___Early_blight/002a55fb-7a3d-4a3a-aca8-ce2d5ebc6925___RS_Early.B 8170.JPG"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean[0], mean[1], mean[2]], std=[std[0], std[1], std[2]])  # use the RGB mean/std you calculated
])

img = Image.open(image_path).convert("RGB")
img_tensor = transform(img).unsqueeze(0).to(device)


In [19]:
with torch.no_grad():
    outputs = model(img_tensor)
    predicted_class = torch.argmax(outputs, dim=1).item()

print(f"Predicted class: {classes[predicted_class]}")


Predicted class: Potato___Early_blight
